# Oversampling

### <i> Using SDV Framework

<br>
<br>

`SDV` learns the overall data distribution and generates synthetic records from that learned distribution. You have a wide variety of options (Synthesizer algorithms) you can work with when deciding how you want to create more data:

</i>
<br>

|Synthesizer|Approach|
|---|---|
|GaussianCopulaSynthesizer| statistical/correlation-based approach (REALLY robust)|
|CTGAN| neural network-based|
|TVAE| variational autoencoder approach (good for noise)|
|CopulaGAN| GAN + copula approach (handles huge ranges in values decently well)|


<i>



<i> Not bad. You can see CTGAN struggle with thin and thick-tailed distributions. You can also see it struggles to mimic the original dataset when big ranges are involved.

#### Gaussian Copula

In [1]:
from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer
import pandas as pd

# Load a new dataframe.
df = pd.read_csv("../../houses.csv")

df.drop(["id", "date", "lat", "long", 'zipcode'], axis=1, inplace=True)

# Create the metadata again
metadata = Metadata.detect_from_dataframe(
    data=df
)

# Place the metadata into the GaussianCopula model approach
synthesizer = GaussianCopulaSynthesizer(
    metadata
)

# Set and fit the data into our dataframe.
synthesizer.fit(df)

# Make 5k rows using Gaussian Copula
synthetic_df = synthesizer.sample(
    num_rows=5000
)

c:\Users\tuomas.valtanen\dataeng_advda2026_lectures\DataEngineering_AdvDA_2026\.venv\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [2]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# # Let's make two histplots for each column. One for the Original dataset, the other for CTGAN.
# for column in df.columns:

#     # Create the canvas and create
#     fig, axes = plt.subplots(1, 2, figsize=(12, 5))
#     sns.histplot(df[column], ax=axes[0], color="red")
#     sns.histplot(synthetic_data[column], ax=axes[1], color="purple")

#     axes[0].set_title("Original")
#     axes[1].set_title("Gaussian Copula Synthetic")



###  Using `SDV` evaluate Quality

In [3]:
# Let's use the evaluate_quality function in SDV's library
from sdv.evaluation.single_table import evaluate_quality

quality_report = evaluate_quality(
    real_data=df,
    synthetic_data=synthetic_df,
    metadata=metadata
)

# The higher the percentile, the better the fit. 
quality_report

# Column shape: Compares real data vs synthetic data distributions.
# Column pairs: Compares pairs, ie real x: income y: age vs. synthetic x: income y:age

# good values are probably in the 80-95% range, we shouldn't expect 100%

C:\Users\tuomas.valtanen\AppData\Local\Temp\ipykernel_13120\3912371679.py:2: FutureWarning: The evaluation functions are now accessible via the 'sdv.evaluation' module.
  from sdv.evaluation.single_table import evaluate_quality


Generating report ...

(1/4) Evaluating Column Shapes: |██████████| 16/16 [00:00<00:00, 227.09it/s]|
Column Shapes Score: 82.79%

(2/4) Evaluating Column Pair Trends: |██████████| 120/120 [00:00<00:00, 365.01it/s]|
Column Pair Trends Score: 95.75%

(3/4) Evaluating Cardinality: N/A
This property does not apply to single-table data.

(4/4) Evaluating Intertable Trends: N/A
This property does not apply to single-table data.

Overall Score (Average): 89.27%



---

In [4]:
# switching to decimal notation for the variance checks
pd.set_option('display.float_format', lambda x: "%.2f" % x)

In [5]:
# We can compare the statistics between the two dataframes by using `.describe()`
df.describe()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,sqft_living15,sqft_lot15
count,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00
mean,540088.14,3.37,2.11,2079.90,15106.97,1.49,0.01,0.23,3.41,7.66,1788.39,291.51,1971.01,84.40,1986.55,12768.46
std,367127.20,0.93,0.77,918.44,41420.51,0.54,0.09,0.77,0.65,1.18,828.09,442.58,29.37,401.68,685.39,27304.18
min,75000.00,0.00,0.00,290.00,520.00,1.00,0.00,0.00,1.00,1.00,290.00,0.00,1900.00,0.00,399.00,651.00
25%,321950.00,3.00,1.75,1427.00,5040.00,1.00,0.00,0.00,3.00,7.00,1190.00,0.00,1951.00,0.00,1490.00,5100.00
50%,450000.00,3.00,2.25,1910.00,7618.00,1.50,0.00,0.00,3.00,7.00,1560.00,0.00,1975.00,0.00,1840.00,7620.00
75%,645000.00,4.00,2.50,2550.00,10688.00,2.00,0.00,0.00,4.00,8.00,2210.00,560.00,1997.00,0.00,2360.00,10083.00
max,7700000.00,33.00,8.00,13540.00,1651359.00,3.50,1.00,4.00,5.00,13.00,9410.00,4820.00,2015.00,2015.00,6210.00,871200.00


In [6]:
# pretty similar!
synthetic_df.describe()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,sqft_living15,sqft_lot15
count,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00
mean,534581.17,3.35,2.11,2038.49,15043.27,2.06,0.01,0.25,3.41,7.64,1776.69,121.80,1970.32,1988.09,1981.84,13102.72
std,289907.76,0.95,0.77,1106.17,15691.49,0.45,0.08,0.79,0.64,1.29,771.48,284.49,29.84,30.55,664.65,12574.42
min,80050.00,1.00,0.20,331.00,521.00,1.50,0.00,0.00,1.00,3.00,484.00,0.00,1900.00,1746.00,606.00,655.00
25%,321472.75,3.00,1.56,1206.75,4188.00,1.70,0.00,0.00,3.00,7.00,1198.00,0.00,1947.00,1977.00,1494.75,4248.25
50%,478316.50,3.00,2.05,1835.00,10055.00,2.00,0.00,0.00,3.00,8.00,1650.00,6.00,1974.00,1998.00,1878.50,9275.00
75%,675408.50,4.00,2.58,2664.25,20259.50,2.40,0.00,0.00,4.00,8.00,2210.75,89.00,1996.00,2009.00,2364.00,17897.00
max,2371682.00,8.00,5.81,7523.00,126214.00,3.50,1.00,4.00,5.00,12.00,5984.00,2636.00,2015.00,2015.00,5341.00,95517.00


In [7]:
synthetic_df

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,sqft_living15,sqft_lot15
0,757458.00,4,3.22,3441,27875,1.80,0,0,3,8,2941,0,1964,2002,2426,27249
1,610858.00,2,2.85,2689,6654,2.80,0,0,4,8,2375,2,2006,1993,2379,11027
2,322130.00,3,2.94,2665,14102,2.30,0,0,3,8,2761,0,2013,1998,2798,8839
3,291785.00,3,1.95,1189,4857,1.80,0,0,5,8,1685,0,1927,1988,2152,7058
4,408479.00,3,1.13,697,1676,2.00,0,0,4,5,878,0,1930,1979,1588,1648
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,211440.00,4,1.76,1373,17911,1.50,0,0,4,7,1275,11,1962,1998,1581,13592
4996,325011.00,3,2.46,1796,936,2.40,0,0,4,7,1111,921,2007,1998,1566,1665
4997,279538.00,2,0.63,607,9922,1.60,0,0,3,5,889,0,1914,1992,1577,12909
4998,1022911.00,3,2.26,2787,52411,1.60,0,0,4,8,1959,157,1957,2000,3543,32097


In [8]:
# hmm, some weird values generated in the yr_renovated
# need post-processing!
df['yr_renovated'].value_counts()

yr_renovated
0       20699
2014       91
2013       37
2003       36
2005       35
        ...  
1948        1
1951        1
1959        1
1934        1
1944        1
Name: count, Length: 70, dtype: int64

In [9]:
synthetic_df['yr_renovated'].value_counts()

yr_renovated
2014    279
2015    235
2013    198
2011    193
2012    188
       ... 
1857      1
1892      1
1874      1
1863      1
1796      1
Name: count, Length: 177, dtype: int64

Study also the variances!

In [10]:
df.var()

price           134782378397.25
bedrooms                   0.87
bathrooms                  0.59
sqft_living           843533.68
sqft_lot          1715658774.18
floors                     0.29
waterfront                 0.01
view                       0.59
condition                  0.42
grade                      1.38
sqft_above            685734.67
sqft_basement         195872.67
yr_built                 862.80
yr_renovated          161346.21
sqft_living15         469761.24
sqft_lot15         745518225.34
dtype: float64

In [ ]:
# some major changes here can be detected
synthetic_df.var()

price           84046509799.10
bedrooms                  0.90
bathrooms                 0.59
sqft_living         1223612.12
sqft_lot          246222733.79
floors                    0.20
waterfront                0.01
view                      0.62
condition                 0.42
grade                     1.66
sqft_above           595185.79
sqft_basement         80933.31
yr_built                890.38
yr_renovated            933.21
sqft_living15        441763.40
sqft_lot15        158115939.88
dtype: float64